# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the FAIR² mass spectrometry EV dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# .metadata is an mlcroissant object, not a dict. Use .name, .description attributes directly
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id
record_sets_info = []
for rs in dataset.metadata.record_sets:
    field_ids = [field['@id'] if isinstance(field, dict) and '@id' in field else str(field) for field in getattr(rs, 'fields', [])]
    record_sets_info.append({
        'record_set_id': rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs),
        'name': getattr(rs, 'name', None),
        'field_ids': field_ids
    })

for info in record_sets_info:
    print(f"RecordSet name: {info['name']} | @id: {info['record_set_id']}")
    print(f"  Fields (@id): {info['field_ids']}")

# If there are no record sets, inform the user
if not record_sets_info:
    print('No record sets available in this dataset (metadata.record_sets is empty).')

In [ ]:
# If at least one RecordSet exists, print some sample records from the first record set.
if dataset.metadata.record_sets:
    first_record_set = dataset.metadata.record_sets[0]
    first_rs_id = first_record_set['@id'] if isinstance(first_record_set, dict) and '@id' in first_record_set else str(first_record_set)
    print(f'Sample records for RecordSet @id: {first_rs_id}')
    for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from available record sets into pandas DataFrames
dataframes = {}
record_set_ids = [info['record_set_id'] for info in record_sets_info] if record_sets_info else []

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        # Only keep if there is any record
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
    except Exception as e:
        print(f'Failed to extract records for {rs_id}: {e}')

if dataframes:
    rs_id = list(dataframes.keys())[0]
    print(f'Columns in DataFrame for record set {rs_id}:')
    print(dataframes[rs_id].columns.tolist())
    display(dataframes[rs_id].head())
else:
    print('No non-empty dataframes were loaded. Make sure the record_sets and records are present.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA on the main record set if available
import numpy as np
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    df = dataframes[main_rs_id]
    print(f"Main record set for EDA: {main_rs_id}")
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].quantile(0.9)  # Use 90th percentile as threshold example
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a non-numeric field if available
        possible_groups = [col for col in df.columns if col != numeric_field and df[col].dtype == 'object']
        if possible_groups:
            group_field = possible_groups[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No non-numeric grouping field found.")
    else:
        print('No numeric fields found for EDA.')
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization example for first numeric field
if dataframes:
    df = dataframes[list(dataframes.keys())[0]]
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        field = numeric_fields[0]
        plt.figure(figsize=(8,5))
        sns.histplot(df[field].dropna(), kde=True, bins=30)
        plt.title(f"Distribution of {field}")
        plt.xlabel(field)
        plt.ylabel("Frequency")
        plt.show()
        
        if len(numeric_fields) > 1:
            plt.figure(figsize=(7,5))
            sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
            plt.title(f"Scatter plot of {numeric_fields[0]} vs {numeric_fields[1]}")
            plt.xlabel(numeric_fields[0])
            plt.ylabel(numeric_fields[1])
            plt.show()
    else:
        print('No numeric fields available for visualization.')
else:
    print('No DataFrames available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we used `mlcroissant` to load, inspect, and perform basic exploratory data analysis (EDA) on the FAIR² mass spectrometry analysis dataset. Using only entity `@id` fields, we programmatically discovered available RecordSets and Fields, extracted data for EDA, and visualized main numeric variables—laying the groundwork for downstream proteomics analysis, further normalization, or advanced machine learning tasks.